# Notebook 03 — Sequence Generation

**Chapter 3, Section 3.5.2 — Sequence Construction**

> *"A sliding window of width 10 was applied across the ordered traffic records,
> constructing input sequences where each sequence contains 10 consecutive network
> connections. The label assigned to each sequence corresponds to the traffic class
> of its final timestep."*

## Objectives
- Convert the 2-D scaled feature matrix to 3-D LSTM sequences
- Verify sequence shapes and label integrity
- Split into train / validation / test sets (70 / 15 / 15)
- Compute and display class weights
- Save all split arrays to `data/processed/`

---

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.utils.helpers import set_global_seed, print_banner
from src.config import get_config
from src.utils.paths import PROCESSED_DATA_DIR

cfg = get_config()
set_global_seed(cfg.seed)
print_banner('Notebook 03 — Sequence Generation')
print(f'Window size : {cfg.sequence.window_size}')
print(f'Step size   : {cfg.sequence.step_size}')
print(f'Label pos   : {cfg.sequence.label_position}')

## 1. Run Full Preprocessing (if not already done)

In [ ]:
from src.data.loaders import load_dataset
from src.data.preprocessing import preprocess_dataset
import json

main_df, _ = load_dataset('nsl_kdd', merge=True, validate=False)

X_scaled, y, scaler, feature_names, metadata = preprocess_dataset(
    main_df,
    dataset='nsl_kdd',
    save_interim_files=False,
    artifacts_dir=PROCESSED_DATA_DIR,
)

n_classes   = metadata['n_classes']
class_names = metadata['class_names']
n_features  = metadata['n_features']

print(f'X_scaled : {X_scaled.shape}  dtype={X_scaled.dtype}')
print(f'y        : {y.shape}  dtype={y.dtype}')
print(f'n_classes: {n_classes}')
print(f'n_features: {n_features}')

## 2. Memory Estimation Before Building

In [ ]:
from src.data.sequence_builder import estimate_sequence_count, estimate_memory_mb

window = cfg.sequence.window_size
n_seqs = estimate_sequence_count(len(X_scaled), window)
est_mb = estimate_memory_mb(n_seqs, window, n_features)

print(f'Input records       : {len(X_scaled):,}')
print(f'Window size         : {window}')
print(f'Estimated sequences : {n_seqs:,}')
print(f'Estimated memory    : {est_mb:.1f} MB')

## 3. Build Sequences (Sliding Window)

In [ ]:
from src.data.sequence_builder import rebuild_sequences_from_flat, get_sequence_stats

X_seq, y_seq = rebuild_sequences_from_flat(
    X_scaled, y,
    window_size=cfg.sequence.window_size,
    step_size=cfg.sequence.step_size,
    label_position=cfg.sequence.label_position,
)

print(f'\nSequence array shapes:')
print(f'  X_seq : {X_seq.shape}  (sequences × window × features)')
print(f'  y_seq : {y_seq.shape}')
print(f'  dtype X: {X_seq.dtype}')
print(f'  dtype y: {y_seq.dtype}')

In [ ]:
stats = get_sequence_stats(X_seq, y_seq, class_names)
print('Sequence statistics:')
for k, v in stats.items():
    if not isinstance(v, dict):
        print(f'  {k:<30} {v}')

print('\nClass distribution in sequences:')
for cls, cnt in stats['class_distribution'].items():
    print(f'  {cls:<12}: {cnt:>8,} ({cnt/stats["n_sequences"]*100:.2f}%)')

## 4. Verify Sequence Content

In [ ]:
# Verify first sequence
print('First sequence (index 0):')
print(f'  Shape      : {X_seq[0].shape}')
print(f'  Label      : {y_seq[0]} ({class_names[y_seq[0]]})')
print(f'  Value range: [{X_seq[0].min():.4f}, {X_seq[0].max():.4f}]')

# Confirm label is from last timestep of original array
expected_label = y[window - 1]
print(f'  Expected label (y[{window-1}]): {expected_label}')
print(f'  Labels match: {y_seq[0] == expected_label}')

## 5. Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
from src.data.split import split_and_save, get_split_summary

X_train, X_val, X_test, y_train, y_val, y_test = split_and_save(
    X_seq, y_seq,
    output_dir=PROCESSED_DATA_DIR,
    train_ratio=cfg.split.train_ratio,
    val_ratio=cfg.split.val_ratio,
    test_ratio=cfg.split.test_ratio,
    stratified=cfg.split.stratified,
    dataset='nsl_kdd',
)

print(f'\nSplit sizes:')
print(f'  X_train : {X_train.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')

In [ ]:
summary = get_split_summary(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    class_names=class_names,
    dataset='nsl_kdd',
)

for split_name, split_data in summary['splits'].items():
    print(f'{split_name.capitalize():<12}: {split_data["n_sequences"]:>7,} sequences '
          f'({split_data["pct_of_total"]:.1f}%)')

## 6. Class Weights for Training

In [ ]:
from src.data.split import compute_split_class_weights

class_weights = compute_split_class_weights(y_train)
print('Class weights (inverse-frequency):')
for cls_int, wt in sorted(class_weights.items()):
    name = class_names[cls_int]
    n_train = int((y_train == cls_int).sum())
    print(f'  {cls_int} ({name:<10}): weight={wt:.4f}  n={n_train:,}')

## 7. Visualise Class Balance Across Splits

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
from src.utils.constants import CLASS_COLORS

for ax, (split_name, y_split) in zip(
    axes,
    [('Train', y_train), ('Validation', y_val), ('Test', y_test)]
):
    unique, counts = np.unique(y_split, return_counts=True)
    bars = ax.bar(
        [class_names[i] for i in unique],
        counts,
        color=[CLASS_COLORS[i] for i in unique],
        edgecolor='white'
    )
    ax.set_title(f'{split_name} ({len(y_split):,} sequences)',
                 fontweight='bold')
    ax.set_ylabel('Sequences')
    ax.tick_params(axis='x', rotation=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Class Distribution Across Splits (Stratified)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/split_class_distribution.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 8. Verify Saved Files

In [ ]:
from src.utils.serialization import load_processed_arrays

X_tr2, X_v2, X_te2, y_tr2, y_v2, y_te2 = \
    load_processed_arrays(PROCESSED_DATA_DIR)

print('Loaded arrays from disk:')
print(f'  X_train : {X_tr2.shape}')
print(f'  X_val   : {X_v2.shape}')
print(f'  X_test  : {X_te2.shape}')
assert np.array_equal(X_tr2, X_train), 'Train arrays do not match!'
print('✓ Saved and reloaded arrays match.')

---
**Summary:**  
- Sliding window sequences built: `X_seq.shape = (N, 10, 41)` where N ≈ dataset size  
- Stratified 70/15/15 split applied — class proportions preserved across all splits  
- Class weights computed (inverse-frequency) for training  
- All arrays saved to `data/processed/`  

**Next:** Run `04_baseline_models.ipynb`